# 딥러닝응용I(추천시스템)

**동덕여자대학교 데이터사이언스전공 유원상 교수**  
**2026년 2학기**

## W01A · 오리엔테이션과 추천시스템의 세계

### 수업 개요
이 notebook은 W01A 수업 중 필수 실습이 아니라 다음 수업 W01B를 위한 짧은 예고입니다.
작은 가상 데이터에서 모든 사용자에게 같은 목록과 사용자별로 다른 목록을 비교합니다.

### 학습목표
1. 사용자, 아이템, 행동 데이터의 예를 표에서 찾는다.
2. 비개인화 추천과 개인화 추천의 출력 차이를 설명한다.
3. 코드를 실행하기 전에 결과를 예측하고 실행 결과와 비교한다.


## 0. 실행 준비

외부 파일이나 네트워크를 사용하지 않습니다. 아래 셀은 표를 보기 위한 pandas만 불러옵니다.


In [ ]:
from __future__ import annotations

import pandas as pd


## 1. 사용자, 아이템, 행동

아래 데이터는 수업을 위해 만든 가상 예시입니다.
한 행은 한 사용자가 한 아이템을 선택한 행동을 나타냅니다.


In [ ]:
interactions = pd.DataFrame(
    [
        {"user_id": "A", "item": "데이터 다큐", "category": "과학"},
        {"user_id": "A", "item": "우주 이야기", "category": "과학"},
        {"user_id": "A", "item": "재즈의 밤", "category": "음악"},
        {"user_id": "B", "item": "도시 여행", "category": "여행"},
        {"user_id": "B", "item": "바다 산책", "category": "여행"},
        {"user_id": "C", "item": "도시 여행", "category": "여행"},
    ]
)
interactions


### 관찰 활동

표를 보고 다음 요소를 찾으세요.

- 사용자: 어느 열에 있는가?
- 아이템: 어느 열에 있는가?
- 행동: 이 표에서 한 행은 무엇을 뜻하는가?
- 맥락 또는 특성: 아이템을 설명하는 추가 정보는 무엇인가?


## 2. 모두에게 같은 목록

가장 많이 선택된 아이템부터 정렬하면 사용자와 관계없이 같은 인기 목록을 만들 수 있습니다.
실행하기 전에 1위가 될 아이템을 예측하세요.


In [ ]:
# 예측을 메모하세요. 이 셀은 정답 검사를 하지 않습니다.
predicted_top_item = None


In [ ]:
popular_items = (
    interactions.groupby("item", as_index=False)
    .size()
    .sort_values(["size", "item"], ascending=[False, True])
    .rename(columns={"size": "selection_count"})
    .reset_index(drop=True)
)
popular_items


이 목록은 누가 요청하더라도 같습니다. 구현이 단순하고 새로운 사용자에게도 보여줄 수 있지만,
사용자별 관심 차이를 직접 반영하지는 않습니다.


## 3. 사용자에 따라 다른 목록

다음 코드는 사용자가 과거에 가장 많이 선택한 category를 찾고, 그 category의 후보를 먼저 보여주는
교육용 규칙입니다. 실제 추천 알고리즘을 대표하는 완성된 모델은 아닙니다.


In [ ]:
candidate_items = pd.DataFrame(
    [
        {"item": "AI 탐험", "category": "과학"},
        {"item": "별빛 관측", "category": "과학"},
        {"item": "골목 여행", "category": "여행"},
        {"item": "섬으로", "category": "여행"},
        {"item": "피아노 오후", "category": "음악"},
    ]
)


def favorite_category(user_id: str) -> str:
    """Return the most frequent category in a user's toy interaction history."""
    user_history = interactions.loc[interactions["user_id"] == user_id, "category"]
    if user_history.empty:
        return "정보 없음"
    return str(user_history.value_counts().index[0])


def recommend_by_category(user_id: str, top_n: int = 2) -> pd.DataFrame:
    """Put candidates from the user's favorite category first."""
    preferred = favorite_category(user_id)
    ranked = candidate_items.assign(is_preferred=candidate_items["category"].eq(preferred))
    return (
        ranked.sort_values(["is_preferred", "item"], ascending=[False, True])
        .head(top_n)
        .drop(columns="is_preferred")
        .reset_index(drop=True)
    )


### 결과 예측

실행하기 전에 생각해 봅시다.

1. 사용자 A에게 먼저 보일 category는 무엇일까요?
2. 사용자 B에게 먼저 보일 category는 무엇일까요?
3. 기록이 없는 사용자 D에게 이 규칙을 적용하면 어떤 한계가 생길까요?


In [ ]:
print("사용자 A의 관심 category:", favorite_category("A"))
display(recommend_by_category("A"))

print("사용자 B의 관심 category:", favorite_category("B"))
display(recommend_by_category("B"))


사용자별 기록을 이용했기 때문에 A와 B의 결과가 달라질 수 있습니다. 그러나 이 간단한 규칙은
category 하나만 사용하며, 선택하지 않은 아이템에 대한 만족도까지 알지는 못합니다.
W01B부터 데이터 열을 직접 확인하고 작은 추천을 실행하면서 이런 가정과 한계를 살펴봅니다.


## 퀴즈

1. 추천시스템이 많은 후보를 다루는 사용자에게 줄 수 있는 도움은 무엇인가요?
2. `popular_items`는 왜 비개인화 추천인가요?
3. `interactions`에서 사용자, 아이템, 아이템 특성에 해당하는 열은 각각 무엇인가요?
4. 한 번의 선택을 사용자의 장기적인 만족이라고 단정하기 어려운 이유는 무엇인가요?
5. `recommend_by_category`가 사용자별로 다른 결과를 만들 수 있는 신호는 무엇인가요?
6. 기록이 전혀 없는 새 사용자에게 이 규칙을 적용할 때 생기는 문제는 무엇인가요?
7. W01B에서 Colab으로 확인하고 싶은 데이터 또는 추천 결과를 한 가지 적어 보세요.


## Take-home message

- 데이터에서 사용자, 아이템, 행동을 구분하면 추천 문제의 기본 구조를 볼 수 있습니다.
- 집계된 인기 목록은 모두에게 같은 결과를 줄 수 있습니다.
- 사용자 행동을 이용하면 결과를 달리할 수 있지만, 데이터와 규칙의 한계를 함께 살펴야 합니다.
- 다음 수업에서는 Colab 환경에서 이 과정을 직접 실행합니다.

## 참고자료

- 임일, 『AI 에이전트를 위한 개인화 추천 알고리즘: Python, 머신러닝, AI, LLM 활용』, 도서출판청람, 2025, 1장.
- 이 notebook의 데이터, 함수와 예시는 수업을 위해 새로 만든 교육용 자료입니다.
